-- IA — Classificação Temática (Rotina Diária) --

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(override=True)
engine = create_engine(os.getenv('DATABASE_URL'))
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('Conexão configurada.')

Conexão configurada.


Temas e embeddings de referência

In [2]:
TEMAS = [
    'Tributação e impostos',
    'Saúde pública e medicina',
    'Educação e ensino',
    'Meio ambiente e sustentabilidade',
    'Trabalho e emprego',
    'Segurança pública e crime',
    'Tecnologia e inovação',
    'Infraestrutura e transporte',
    'Economia e finanças públicas',
    'Direitos humanos e cidadania'
]

response = client.embeddings.create(model='text-embedding-3-small', input=TEMAS)
embeddings_temas = [item.embedding for item in response.data]

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def classificar(ementa):
    resp = client.embeddings.create(model='text-embedding-3-small', input=str(ementa))
    sims = [cosine_similarity(resp.data[0].embedding, et) for et in embeddings_temas]
    return TEMAS[int(np.argmax(sims))]

print('Temas carregados.')

Temas carregados.


Teste — 10 Proposições

In [3]:
df_teste = pd.read_sql('SELECT id, ementa FROM proposicoes LIMIT 10', engine)

for _, row in df_teste.iterrows():
    tema = classificar(row['ementa'])
    print(f'[{tema}]')
    print(f'  {str(row["ementa"])[:100]}...')
    print()

[Infraestrutura e transporte]
  Institui o dia 02 de julho como data histórica no calendário das efemérides nacionais e concede a de...



[Tributação e impostos]
  Modifica a Lei nº 8.078, de 1990, vedando a cobrança de ligação telefônica a serviço de atendimento ...



[Segurança pública e crime]
  Altera a Lei nº 9.503, de 23 de setembro de 1997, que "Institui o Código de Trânsito Brasileiro", pa...



[Segurança pública e crime]
  Altera os arts. 302 e 303 da Lei nº 9.503, de 23 de setembro de 1997 (Código de Trânsito Brasileiro)...



[Segurança pública e crime]
  Altera a Lei nº 9.503, de 23 de setembro de 1997, que institui o Código de Trânsito Brasileiro, para...



[Tributação e impostos]
  Estabelece limite para a taxa de juros praticada por instituições financeiras nacionais a pessoas fí...



[Tributação e impostos]
  Assegura a concessão de aposentadoria especial após  vinte  e cinco  anos  de contribuição  aos  mot...



[Segurança pública e crime]
  Altera a redação dos §§ 1º e 2º do art. 126 da Lei nº 8.213, de 24 de julho de 1991 e dá outras prov...



[Segurança pública e crime]
  Acrescenta incisos aos arts. 44, 89 e 128 da Lei Complementar nº 80, de 12 de janeiro de 1994, para ...



[Economia e finanças públicas]
  Obriga a Caixa Econômica Federal a divulgar os premiados nas loterias que administra....



Classificar novas proposições (tema IS NULL)

In [4]:
from IPython.display import clear_output

with engine.connect() as conn:
    conn.execute(text('ALTER TABLE proposicoes ADD COLUMN IF NOT EXISTS tema TEXT'))
    conn.commit()

df = pd.read_sql('SELECT id, ementa FROM proposicoes WHERE tema IS NULL AND ementa IS NOT NULL', engine)
print(f'{len(df)} proposições para classificar')

with engine.connect() as conn:
    for i, (_, row) in enumerate(df.iterrows()):
        try:
            tema = classificar(row['ementa'])
            conn.execute(
                text('UPDATE proposicoes SET tema = :tema WHERE id = :id'),
                {'tema': tema, 'id': int(row['id'])}
            )
            conn.commit()
            clear_output(wait=True)
            print(f'{i + 1}/{len(df)} classificadas')
        except Exception as e:
            print(f'Erro na proposição {row["id"]}: {e}')

print('Classificação concluída.')

1053/1053 classificadas
Classificação concluída.
